# 多路召回（All）


In [ ]:
import os, math, warnings, pickle, random
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import logging
warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS'] = '1'
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

import faiss
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder


In [ ]:
# 数据路径（独立项目版：纯相对路径）
project_root = Path.cwd().resolve()
data_path = project_root / 'data' / 'raw' / 'news_recommendation'
save_path = project_root / 'data' / 'processed' / 'temp_results'
output_path = project_root / 'outputs'

for path in [data_path, save_path, output_path]:
    path.mkdir(parents=True, exist_ok=True)

print(f"📁 项目根目录: {project_root}")
print(f"📁 原始数据路径: {data_path}")
print(f"📁 中间结果路径: {save_path}")
print(f"📁 导出结果路径: {output_path}")

# ============================================================================
# 🎯 核心配置：选择运行模式
# ============================================================================
# "offline" = 全用户 leave-last-out 离线验证模式
#             = 每个用户保留最后一次点击作为 target，再按用户切分 rerank train/val
# "online"  = 在线提交模式：使用全量训练集 + 测试集历史，生成最终召回结果
# ============================================================================
MODE = "offline"  # 👈 修改这里切换模式！

# 调试模式：使用采样数据加速调试
DEBUG_MODE = False
DEBUG_SAMPLE_USERS = 10000  # 调试时采样的用户数
RANDOM_SEED = 2020
RERANK_VAL_RATIO = 0.2

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

FINAL_RECALL_DICT_NAME = 'offline_all_final_recall_items_dict.pkl' if MODE == "offline" else 'online_all_final_recall_items_dict.pkl'
FINAL_RECALL_CSV_NAME = 'final_recall_for_rank_all.csv'

# 根据 MODE 自动设置 metric_recall 标志（保持向后兼容）
metric_recall = (MODE == "offline")

print("=" * 60)
print(f"🔧 当前运行模式: {MODE}")
print(f"🔧 调试模式: {'开启' if DEBUG_MODE else '关闭'}")
print(f"🎲 随机种子: {RANDOM_SEED}")
if MODE == "offline":
    print("🔬 离线验证模式: 全用户 leave-last-out + rerank 用户切分")
    print(f"   📐 rerank 验证用户比例: {RERANK_VAL_RATIO}")
else:
    print("🚀 在线提交模式: 使用全量数据，生成最终召回结果")
print("=" * 60)


## 读取数据

### ⚠️ 数据泄漏警告

在**离线验证模式**下，必须特别注意数据泄漏问题：

| 数据 | 离线模式用途 | 在线模式用途 |
|------|-------------|-------------|
| `click_trn` | 训练召回模型的数据（不含验证用户） | 训练数据 |
| `click_val` | 验证用户的历史点击（不含最后一次） | - |
| `val_ans` | 验证用户的最后一次点击（ground truth） | - |
| `click_tst` | - | 测试集数据 |

**关键原则**：
1. 计算相似度矩阵时，只能用 `click_trn`，不能用 `click_val` 或 `val_ans`
2. 为验证用户召回时，只能用该用户在 `click_val` 中的历史，不能看到 `val_ans`
3. 评估时才用 `val_ans` 作为 ground truth

In [ ]:
# debug模式： 从训练集中划出一部分数据来调试代码
def get_all_click_sample(data_path, sample_nums=10000):
    """
        训练集中采样一部分数据调试
        data_path: 原数据的存储路径
        sample_nums: 采样数目（这里由于机器的内存限制，可以采样用户做）
    """
    all_click = pd.read_csv(data_path / 'train_click_log.csv')
    all_user_ids = all_click.user_id.unique()

    sample_user_ids = np.random.choice(all_user_ids, size=sample_nums, replace=False) 
    all_click = all_click[all_click['user_id'].isin(sample_user_ids)]
    
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click


def get_all_click_df(data_path, mode="offline"):
    """
    读取点击数据
    
    参数:
        data_path: 数据路径
        mode: 运行模式
            - "offline": 离线验证模式，只使用训练集
            - "online": 在线提交模式，合并训练集和测试集
    
    返回:
        all_click: 点击数据DataFrame
    """
    if mode == "offline":
        # ═══════════════════════════════════════════════════════════════════
        # 🔬 离线验证模式: 只使用训练集
        # ═══════════════════════════════════════════════════════════════════
        print("📊 [离线模式] 加载训练集数据...")
        all_click = pd.read_csv(data_path / 'train_click_log.csv')
    else:
        # ═══════════════════════════════════════════════════════════════════
        # 🚀 在线提交模式: 合并训练集和测试集
        # ═══════════════════════════════════════════════════════════════════
        print("📊 [在线模式] 加载训练集 + 测试集数据...")
        trn_click = pd.read_csv(data_path / 'train_click_log.csv')
        tst_click = pd.read_csv(data_path / 'testA_click_log.csv')
        all_click = pd.concat([trn_click, tst_click]).reset_index(drop=True)
        print(f"   训练集: {len(trn_click)} 条, 测试集: {len(tst_click)} 条")
    
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    print(f"   去重后总数据量: {len(all_click)} 条")
    return all_click


# ============================================================================
# 加载数据（根据当前 MODE 自动选择）
# ============================================================================
all_click_df = get_all_click_df(data_path, mode=MODE)

In [ ]:
# ============================================================================
# 🔬 All 协议：全用户 leave-last-out + rerank 用户切分
# ============================================================================
# ⚠️ 数据泄漏防护：
#    - click_hist_all: 每个用户去掉最后一次点击后的历史
#    - click_last_all: 每个用户最后一次点击（只用于评估和打标签）
#    - rank_user_split: 仅供 rerank 层做 train/val 用户切分，Recall 不再按用户拆分
# ============================================================================

def leave_last_click_out(all_click_df):
    """对每个用户保留最后一次点击作为 target，其余作为历史。"""
    all_click = all_click_df.sort_values(['user_id', 'click_timestamp']).copy()
    click_last_df = all_click.groupby('user_id').tail(1).copy()
    click_hist_df = (
        all_click.groupby('user_id', group_keys=False)
        .apply(lambda x: x.iloc[:-1])
        .reset_index(drop=True)
    )

    # 去除只有一条点击的用户，保证 hist/last 成对存在
    valid_users = click_hist_df['user_id'].unique()
    click_last_df = click_last_df[click_last_df['user_id'].isin(valid_users)].reset_index(drop=True)
    click_hist_df = click_hist_df[click_hist_df['user_id'].isin(click_last_df['user_id'].unique())].reset_index(drop=True)

    return click_hist_df, click_last_df


def split_rerank_users(user_ids, val_ratio=0.2, seed=2020):
    """按用户切分 rerank train/val，仅用于排序阶段调参。"""
    user_ids = np.array(sorted(pd.unique(user_ids)))
    rng = np.random.RandomState(seed)
    rng.shuffle(user_ids)

    if len(user_ids) <= 1:
        return user_ids.tolist(), []

    val_size = int(len(user_ids) * val_ratio)
    val_size = max(val_size, 1)
    val_size = min(val_size, len(user_ids) - 1)

    val_users = user_ids[:val_size].tolist()
    train_users = user_ids[val_size:].tolist()
    return train_users, val_users


In [ ]:
# ============================================================================
# 根据模式进行数据划分
# ============================================================================

max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))

if MODE == "offline":
    # ═══════════════════════════════════════════════════════════════════════
    # 🔬 离线验证模式：全用户 leave-last-out
    # ═══════════════════════════════════════════════════════════════════════
    print("=" * 60)
    print("🔬 [离线模式] 全用户 leave-last-out 划分...")
    print("   ⚠️ 每个用户只隐藏最后一次点击作为 target")
    print("=" * 60)

    all_click_df['click_timestamp'] = all_click_df[['click_timestamp']].apply(max_min_scaler)
    click_hist_all, click_last_all = leave_last_click_out(all_click_df)

    click_tst = pd.read_csv(data_path / 'testA_click_log.csv')
    click_tst['click_timestamp'] = click_tst[['click_timestamp']].apply(max_min_scaler)

    train_users, val_users = split_rerank_users(
        click_hist_all['user_id'].unique(),
        val_ratio=RERANK_VAL_RATIO,
        seed=RANDOM_SEED,
    )
    rank_user_split = {
        'train_users': [int(u) for u in train_users],
        'val_users': [int(u) for u in val_users],
        'seed': RANDOM_SEED,
        'ratio': RERANK_VAL_RATIO,
    }

    # Recall 阶段统一使用全用户历史，不再按用户切开
    click_trn = click_hist_all.copy()
    click_val = None
    val_ans = click_last_all.copy()
    click_data_for_sim = click_hist_all
    trn_last_click_df = click_last_all

    click_hist_all.to_csv(save_path / 'click_hist_all.csv', index=False)
    click_last_all.to_csv(save_path / 'click_last_all.csv', index=False)
    click_tst.to_csv(save_path / 'click_tst_all.csv', index=False)
    pickle.dump(rank_user_split, open(save_path / 'rank_user_split_all.pkl', 'wb'))

    print(f"✅ 历史点击用户数: {click_hist_all['user_id'].nunique()}")
    print(f"✅ 目标点击数: {len(click_last_all)}")
    print(f"✅ rerank 训练用户数: {len(train_users)}")
    print(f"✅ rerank 验证用户数: {len(val_users)}")
    print(f"✅ 测试集用户数: {click_tst['user_id'].nunique()}")

else:
    # ═══════════════════════════════════════════════════════════════════════
    # 🚀 在线提交模式：使用全量数据
    # ═══════════════════════════════════════════════════════════════════════
    print("=" * 60)
    print("🚀 [在线模式] 使用全量数据，跳过离线验证切分")
    print("=" * 60)

    click_tst = pd.read_csv(data_path / 'testA_click_log.csv')
    click_tst['click_timestamp'] = click_tst[['click_timestamp']].apply(max_min_scaler)

    click_trn = all_click_df.copy()
    click_trn['click_timestamp'] = click_trn[['click_timestamp']].apply(max_min_scaler)
    click_data_for_sim = click_trn

    click_trn.to_csv(save_path / 'click_trn_all.csv', index=False)
    click_tst.to_csv(save_path / 'click_tst_all.csv', index=False)

    click_hist_all = None
    click_last_all = None
    click_val = None
    val_ans = None
    rank_user_split = None
    trn_last_click_df = None

    print(f"✅ 训练数据量: {len(click_trn)}")
    print(f"✅ 测试集用户数: {click_tst['user_id'].nunique()}")


In [ ]:
# 读取文章的基本属性
def get_item_info_df(data_path):
    item_info_df = pd.read_csv(data_path / 'articles.csv')
    # 为了方便与训练集中的click_article_id拼接，需要把article_id修改成click_article_id
    item_info_df = item_info_df.rename(columns={'article_id': 'click_article_id'})
    return item_info_df


In [ ]:
# 读取文章的Embedding数据
def get_item_emb_dict(data_path):
    item_emb_df = pd.read_csv(data_path / 'articles_emb.csv')
    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols])
    # 进行归一化
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)
    item_emb_dict = dict(zip(item_emb_df['article_id'], item_emb_np))
    pickle.dump(item_emb_dict, open(save_path / 'item_content_emb_all.pkl', 'wb'))
    return item_emb_dict


In [ ]:
# #采样数据
# all_click_df = get_all_click_df(data_path, offline=True)
# # 全量训练集
# # 对时间戳进行归一化,用于在关联规则的时候计算权重
# all_click_df['click_timestamp'] = all_click_df[['click_timestamp']].apply(max_min_scaler)


In [ ]:
item_info_df = get_item_info_df(data_path)


In [ ]:
item_emb_dict = get_item_emb_dict(data_path)


## 工具函数

### 获取用户-文章-时间函数

这个在基于关联规则的用户协同过滤的时候会用到

In [ ]:
def print_recall_details(user_recall_items_dict, trn_last_click_df, item_info_df, topk=10, num_samples=3):
    """
    打印详细的召回结果：包括用户真实点击、召回文章详情、是否命中等
    """
    import random
    print(f"\n========== 召回详情查看 (Top-{topk}) ==========")
    
    # 建立用户最后一次点击的字典（用于对比是否命中）
    last_click_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    
    # 建立文章详情字典（用于显示文章标题/类别等）
    # 假设 item_info_df 已经加载，且有 click_article_id 列
    if 'click_article_id' in item_info_df.columns:
        item_info_dict = item_info_df.set_index('click_article_id').to_dict('index')
    else:
        item_info_dict = {}
    
    # 随机采样几个用户进行展示
    available_users = list(user_recall_items_dict.keys())
    if not available_users:
        print("当前召回字典为空")
        return
        
    sample_users = random.sample(available_users, min(num_samples, len(available_users)))
    
    for user in sample_users:
        print(f"\n----------- 用户ID: {user} -----------")
        true_item = last_click_dict.get(user, -1)
        print(f"【真实点击】: {true_item}")
        if true_item in item_info_dict:
            print(f"   └─ 详情: {item_info_dict[true_item]}")
            
        print(f"【召回列表】:")
        # 获取该用户的召回列表
        items = user_recall_items_dict[user]
        # 兼容两种格式：[(item, score), ...] 或 {item: score}
        if isinstance(items, dict):
            items = sorted(items.items(), key=lambda x:x[1], reverse=True)
        
        # 打印前 TopK 个
        for idx, (item, score) in enumerate(items[:topk]):
            mark = "✅ [HIT]" if item == true_item else "   "
            info = item_info_dict.get(item, "无详情")
            print(f"   {idx+1}. {mark} 文章ID: {item:<6} 分数: {score:.4f}Info: {info}")
    print("==============================================\n")

In [ ]:
# 根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
def get_user_item_time(click_df):
    
    click_df = click_df.sort_values('click_timestamp')
    
    def make_item_time_pair(df):
        return list(zip(df['click_article_id'], df['click_timestamp']))
    
    user_item_time_df = click_df.groupby('user_id')[['click_article_id', 'click_timestamp']].apply(lambda x: make_item_time_pair(x))\
                                                            .reset_index().rename(columns={0: 'item_time_list'})
    user_item_time_dict = dict(zip(user_item_time_df['user_id'], user_item_time_df['item_time_list']))
    
    return user_item_time_dict


### 获取文章-用户-时间函数

这个在基于关联规则的文章协同过滤的时候会用到

In [ ]:
# 根据时间获取商品被点击的用户序列  {item1: {user1: time1, user2: time2...}...}
# 这里的时间是用户点击当前商品的时间，好像没有直接的关系。
def get_item_user_time_dict(click_df):
    def make_user_time_pair(df):
        return list(zip(df['user_id'], df['click_timestamp']))
    
    click_df = click_df.sort_values('click_timestamp')
    item_user_time_df = click_df.groupby('click_article_id')[['user_id', 'click_timestamp']].apply(lambda x: make_user_time_pair(x))\
                                                            .reset_index().rename(columns={0: 'user_time_list'})
    
    item_user_time_dict = dict(zip(item_user_time_df['click_article_id'], item_user_time_df['user_time_list']))
    return item_user_time_dict


### 获取历史和最后一次点击

这个在评估召回结果， 特征工程和制作标签转成监督学习测试集的时候回用到

In [ ]:
# 获取当前数据的历史点击和最后一次点击
def get_hist_and_last_click(all_click):
    
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df


### 获取文章属性特征

In [ ]:
# 获取文章id对应的基本属性，保存成字典的形式，方便后面召回阶段，冷启动阶段直接使用
def get_item_info_dict(item_info_df):
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts'] = item_info_df[['created_at_ts']].apply(max_min_scaler)
    
    item_type_dict = dict(zip(item_info_df['click_article_id'], item_info_df['category_id']))
    item_words_dict = dict(zip(item_info_df['click_article_id'], item_info_df['words_count']))
    item_created_time_dict = dict(zip(item_info_df['click_article_id'], item_info_df['created_at_ts']))
    
    return item_type_dict, item_words_dict, item_created_time_dict


### 获取用户历史点击的文章信息

In [ ]:
def get_user_hist_item_info_dict(all_click):
    
    # 获取user_id对应的用户历史点击文章类型的集合字典
    user_hist_item_typs = all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_typs_dict = dict(zip(user_hist_item_typs['user_id'], user_hist_item_typs['category_id']))
    
    # 获取user_id对应的用户点击文章的集合
    user_hist_item_ids_dict = all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict = dict(zip(user_hist_item_ids_dict['user_id'], user_hist_item_ids_dict['click_article_id']))
    
    # 获取user_id对应的用户历史点击的文章的平均字数字典
    user_hist_item_words = all_click.groupby('user_id')['words_count'].agg('mean').reset_index()
    user_hist_item_words_dict = dict(zip(user_hist_item_words['user_id'], user_hist_item_words['words_count']))
    
    # 获取user_id对应的用户最后一次点击的文章的创建时间
    all_click_ = all_click.sort_values('click_timestamp')
    user_last_item_created_time = all_click_.groupby('user_id')['created_at_ts'].apply(lambda x: x.iloc[-1]).reset_index()
    
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    user_last_item_created_time['created_at_ts'] = user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    
    user_last_item_created_time_dict = dict(zip(user_last_item_created_time['user_id'], \
                                                user_last_item_created_time['created_at_ts']))
    
    return user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict


### 获取点击次数最多的Top-k个文章

In [ ]:
# 获取近期点击最多的文章
def get_item_topk_click(click_df, k):
    topk_click = click_df['click_article_id'].value_counts().index[:k]
    return topk_click


### 定义多路召回字典

In [ ]:
# 获取文章的属性信息，保存成字典的形式方便查询
item_type_dict, item_words_dict, item_created_time_dict = get_item_info_dict(item_info_df)


In [ ]:
# 定义一个多路召回的字典，将各路召回的结果都保存在这个字典当中
user_multi_recall_dict = {
    'itemcf_sim_itemcf_recall': {},
    'embedding_sim_item_recall': {},
    'cold_start_recall': {},
}


In [ ]:
# ============================================================================
# 🔬 设置召回目标用户和评估用的 Ground Truth
# ============================================================================
# 离线模式：
#   - 召回目标：全用户历史 click_hist_all
#   - 评估 Ground Truth：click_last_all
# 在线模式：
#   - 召回目标：测试用户
#   - 无需评估
# ============================================================================

if MODE == "offline":
    trn_hist_click_df = click_hist_all
    trn_last_click_df = click_last_all
    print(f"🔬 [离线模式] 召回目标: 全用户 leave-last-out")
    print(f"   召回用户数: {trn_hist_click_df['user_id'].nunique()}")
    print(f"   评估样本数: {len(trn_last_click_df)}")
else:
    trn_hist_click_df = click_tst
    trn_last_click_df = None
    print(f"🚀 [在线模式] 召回目标: 测试用户")
    print(f"   测试用户数: {trn_hist_click_df['user_id'].nunique()}")


### 召回效果评估

做完了召回有时候也需要对当前的召回方法或者参数进行调整以达到更好的召回效果，因为召回的结果决定了最终排序的上限，下面也会提供一个召回评估的方法

In [ ]:
# 依次评估召回的前10, 20, 30, 40, 50个文章中的击中率
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=5):
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    user_num = len(user_recall_items_dict)
    
    for k in range(10, topk+1, 10):
        hit_num = 0
        eval_user_num = 0  # 实际评估的用户数
        for user, item_list in user_recall_items_dict.items():
            # 跳过不在验证集中的用户
            if user not in last_click_item_dict:
                continue
            eval_user_num += 1
            # 获取前k个召回的结果
            tmp_recall_items = [x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp_recall_items):
                hit_num += 1
        
        if eval_user_num > 0:
            hit_rate = round(hit_num * 1.0 / eval_user_num, 5)
            print(' topk: ', k, ' : ', 'hit_num: ', hit_num, 'hit_rate: ', hit_rate, 'eval_user_num : ', eval_user_num)
        else:
            print(' topk: ', k, ' : 没有可评估的用户')

## 计算相似性矩阵

这一部分主要是通过协同过滤以及向量检索得到相似性矩阵，相似性矩阵主要分为user2user和item2item，下面依次获取基于itemCF的item2item的相似性矩阵。

### ⚠️ 数据泄漏防护

| 模式 | 用于计算相似度的数据 | 说明 |
|------|---------------------|------|
| 离线模式 | `click_trn` (仅训练用户) | ⚠️ 不能包含验证用户的任何数据 |
| 在线模式 | `all_click_df` (全量数据) | 包含训练集+测试集 |

### itemCF i2i_sim

借鉴KDD2020的去偏商品推荐，在计算item2item相似性矩阵时，使用关联规则，使得计算的文章的相似性还考虑到了:
1. 用户点击的时间权重
2. 用户点击的顺序权重
3. 文章创建的时间权重

In [ ]:
def itemcf_sim(df, item_created_time_dict):
    """
        文章与文章之间的相似性矩阵计算
        :param df: 数据表
        :item_created_time_dict:  文章创建时间的字典
        return : 文章与文章的相似性矩阵

        思路: 基于物品的协同过滤(详细请参考上一期推荐系统基础的组队学习) + 关联规则
    """

    user_item_time_dict = get_user_item_time(df)

    # 计算物品相似度
    i2i_sim = {}
    item_cnt = defaultdict(int)
    for user, item_time_list in tqdm(user_item_time_dict.items(), disable=not logger.isEnabledFor(logging.DEBUG)):
        # 在基于商品的协同过滤优化的时候可以考虑时间因素
        for loc1, (i, i_click_time) in enumerate(item_time_list):
            item_cnt[i] += 1
            i2i_sim.setdefault(i, {})
            for loc2, (j, j_click_time) in enumerate(item_time_list):
                if(i == j):
                    continue

                # 考虑文章的正向顺序点击和反向顺序点击    
                loc_alpha = 1.0 if loc2 > loc1 else 0.7
                # 位置信息权重，其中的参数可以调节
                loc_weight = loc_alpha * (0.9 ** (np.abs(loc2 - loc1) - 1))
                # 点击时间权重，其中的参数可以调节
                click_time_weight = np.exp(0.7 ** np.abs(i_click_time - j_click_time))
                # 两篇文章创建时间的权重，其中的参数可以调节
                created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                i2i_sim[i].setdefault(j, 0)
                # 考虑多种因素的权重计算最终的文章之间的相似度
                i2i_sim[i][j] += loc_weight * click_time_weight * created_time_weight / math.log(len(item_time_list) + 1)

    i2i_sim_ = i2i_sim.copy()
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])

    # 将得到的相似性矩阵保存到本地
    pickle.dump(i2i_sim_, open(save_path / 'itemcf_i2i_sim_all.pkl', 'wb'))

    return i2i_sim_


In [ ]:
# ============================================================================
# 计算 ItemCF 相似度矩阵
# ============================================================================
# ⚠️ 离线模式：用 click_data_for_sim（训练用户 + 验证用户历史，不含 val_ans）
# ⚠️ 在线模式：用全量数据
# ============================================================================

print("=" * 60)
if MODE == "offline":
    print("🔬 [离线模式] 计算 ItemCF 相似度...")
    print("   ✅ 使用训练用户 + 验证用户历史（不含 val_ans，无数据泄漏）")
    sim_data = click_data_for_sim  # 训练用户 + 验证用户历史
else:
    print("🚀 [在线模式] 计算 ItemCF 相似度...")
    print("   使用全量数据")
    sim_data = click_trn

i2i_sim = itemcf_sim(sim_data, item_created_time_dict)
print(f"✅ ItemCF 相似度矩阵计算完成，包含 {len(i2i_sim)} 个物品")
print("=" * 60)

### userCF u2u_sim

在计算用户之间的相似度的时候，也可以使用一些简单的关联规则，比如用户活跃度权重，这里将用户的点击次数作为用户活跃度的指标

In [ ]:
def get_user_activate_degree_dict(all_click_df):
    all_click_df_ = all_click_df.groupby('user_id')['click_article_id'].count().reset_index()
    
    # 用户活跃度归一化
    mm = MinMaxScaler()
    all_click_df_['click_article_id'] = mm.fit_transform(all_click_df_[['click_article_id']])
    user_activate_degree_dict = dict(zip(all_click_df_['user_id'], all_click_df_['click_article_id']))
    
    return user_activate_degree_dict


In [ ]:
def usercf_sim(all_click_df, user_activate_degree_dict):
    """
        用户相似性矩阵计算
        :param all_click_df: 数据表
        :param user_activate_degree_dict: 用户活跃度的字典
        return 用户相似性矩阵

        思路: 基于用户的协同过滤(详细请参考上一期推荐系统基础的组队学习) + 关联规则
    """
    item_user_time_dict = get_item_user_time_dict(all_click_df)

    u2u_sim = {}
    user_cnt = defaultdict(int)
    for item, user_time_list in tqdm(item_user_time_dict.items(), disable=not logger.isEnabledFor(logging.DEBUG)):
        for u, click_time in user_time_list:
            user_cnt[u] += 1
            u2u_sim.setdefault(u, {})
            for v, click_time in user_time_list:
                u2u_sim[u].setdefault(v, 0)
                if u == v:
                    continue
                # 用户平均活跃度作为活跃度的权重，这里的式子也可以改善
                activate_weight = 100 * 0.5 * (user_activate_degree_dict[u] + user_activate_degree_dict[v])   
                u2u_sim[u][v] += activate_weight / math.log(len(user_time_list) + 1)

    u2u_sim_ = u2u_sim.copy()
    for u, related_users in u2u_sim.items():
        for v, wij in related_users.items():
            u2u_sim_[u][v] = wij / math.sqrt(user_cnt[u] * user_cnt[v])

    # 将得到的相似性矩阵保存到本地
    pickle.dump(u2u_sim_, open(save_path / 'usercf_u2u_sim_all.pkl', 'wb'))

    return u2u_sim_


In [ ]:
# 由于usercf计算时候太耗费内存了，这里就不直接运行了
# 如果是采样的话，是可以运行的
#user_activate_degree_dict = get_user_activate_degree_dict(all_click_df)
#u2u_sim = usercf_sim(all_click_df, user_activate_degree_dict)


### item embedding sim

使用Embedding计算item之间的相似度是为了后续冷启动的时候可以获取未出现在点击数据中的文章，后面有对冷启动专门的介绍，这里简单的说一下faiss。

aiss是Facebook的AI团队开源的一套用于做聚类或者相似性搜索的软件库，底层是用C++实现。Faiss因为超级优越的性能，被广泛应用于推荐相关的业务当中.

faiss工具包一般使用在推荐系统中的向量召回部分。在做向量召回的时候要么是u2u,u2i或者i2i，这里的u和i指的是user和item.我们知道在实际的场景中user和item的数量都是海量的，我们最容易想到的基于向量相似度的召回就是使用两层循环遍历user列表或者item列表计算两个向量的相似度，但是这样做在面对海量数据是不切实际的，faiss就是用来加速计算某个查询向量最相似的topk个索引向量。

**faiss查询的原理：**

faiss使用了PCA和PQ(Product quantization乘积量化)两种技术进行向量压缩和编码，当然还使用了其他的技术进行优化，但是PCA和PQ是其中最核心部分。

1. PCA降维算法细节参考下面这个链接进行学习    
[主成分分析（PCA）原理总结](https://www.cnblogs.com/pinard/p/6239403.html)  

2. PQ编码的细节下面这个链接进行学习    
[实例理解product quantization算法](http://www.fabwrite.com/productquantization)

**faiss使用**

[faiss官方教程](https://github.com/facebookresearch/faiss/wiki/Getting-started)

In [ ]:
# 向量检索相似度计算
# topk指的是每个item, faiss搜索后返回最相似的topk个item
def embdding_sim(click_df, item_emb_df, save_path, topk):
    """
        基于内容的文章embedding相似性矩阵计算
        :param click_df: 数据表
        :param item_emb_df: 文章的embedding
        :param save_path: 保存路径
        :patam topk: 找最相似的topk篇
        return 文章相似性矩阵

        思路: 对于每一篇文章， 基于embedding的相似性返回topk个与其最相似的文章， 只不过由于文章数量太多，这里用了faiss进行加速
    """

    # 文章索引与文章id的字典映射
    item_idx_2_rawid_dict = dict(zip(item_emb_df.index, item_emb_df['article_id']))

    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols].values, dtype=np.float32)
    # 向量进行单位化
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)

    # 建立faiss索引
    item_index = faiss.IndexFlatIP(item_emb_np.shape[1])
    item_index.add(item_emb_np)
    # 相似度查询，给每个索引位置上的向量返回topk个item以及相似度
    sim, idx = item_index.search(item_emb_np, topk) # 返回的是列表

    # 将向量检索的结果保存成原始id的对应关系
    item_sim_dict = defaultdict(dict)
    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(range(len(item_emb_np)), sim, idx)):
        target_raw_id = item_idx_2_rawid_dict[target_idx]
        # 从1开始是为了去掉商品本身, 所以最终获得的相似商品只有topk-1
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]): 
            rele_raw_id = item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id] = item_sim_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value

    # 保存i2i相似度矩阵
    pickle.dump(item_sim_dict, open(save_path / 'emb_i2i_sim_all.pkl', 'wb'))   

    return item_sim_dict


In [ ]:
# ============================================================================
# 计算 Embedding 相似度矩阵
# ============================================================================
# ⚠️ 离线模式：用 click_data_for_sim（训练用户 + 验证用户历史，不含 val_ans）
# ⚠️ 在线模式：用全量数据
# ============================================================================

print("=" * 60)
if MODE == "offline":
    print("🔬 [离线模式] 计算 Embedding 相似度...")
    print("   ✅ 使用训练用户 + 验证用户历史（不含 val_ans，无数据泄漏）")
    sim_data = click_data_for_sim  # 训练用户 + 验证用户历史
else:
    print("🚀 [在线模式] 计算 Embedding 相似度...")
    sim_data = click_trn

item_emb_df = pd.read_csv(data_path / 'articles_emb.csv').reset_index(drop=True)
emb_i2i_sim = embdding_sim(sim_data, item_emb_df, save_path, topk=10)
print(f"✅ Embedding 相似度矩阵计算完成")
print("=" * 60)

## 召回

这个就是我们开篇提到的那个问题， 面的36万篇文章， 20多万用户的推荐， 我们又有哪些策略来缩减问题的规模？ 我们就可以再召回阶段筛选出用户对于点击文章的候选集合， 从而降低问题的规模。召回常用的策略：
* Youtube DNN 召回
* 基于文章的召回
    * 文章的协同过滤
    * 基于文章embedding的召回
* 基于用户的召回
    * 用户的协同过滤
    * 用户embedding

上面的各种召回方式一部分在基于用户已经看得文章的基础上去召回与这些文章相似的一些文章， 而这个相似性的计算方式不同， 就得到了不同的召回方式， 比如文章的协同过滤， 文章内容的embedding等。还有一部分是根据用户的相似性进行推荐，对于某用户推荐与其相似的其他用户看过的文章，比如用户的协同过滤和用户embedding。 还有一种思路是类似矩阵分解的思路，先计算出用户和文章的embedding之后，就可以直接算用户和文章的相似度， 根据这个相似度进行推荐， 比如YouTube DNN。 我们下面详细来看一下每一个召回方法：



### YoutubeDNN召回

**(这一步是直接获取用户召回的候选文章列表)**

[论文下载地址](https://static.googleusercontent.com/media/research.google.com/zh-CN//pubs/archive/45530.pdf)

**Youtubednn召回架构**

![YoutubeDNN召回架构](../img/youtubednn_candidate.png)



关于YoutubeDNN原理和应用推荐看王喆的两篇博客：

1. [重读Youtube深度学习推荐系统论文，字字珠玑，惊为神文](https://zhuanlan.zhihu.com/p/52169807)
2. [YouTube深度学习推荐系统的十大工程问题](https://zhuanlan.zhihu.com/p/52504407)


**参考文献:**
1. https://zhuanlan.zhihu.com/p/52169807 (YouTubeDNN原理)
2. https://zhuanlan.zhihu.com/p/26306795 (Word2Vec知乎众赞文章) --- word2vec放到排序中的w2v的介绍部分

In [ ]:
# 获取双塔召回时的训练验证数据
# negsample指的是通过滑窗构建样本的时候，负样本的数量
def gen_data_set(data, negsample=0):
    data.sort_values("click_timestamp", inplace=True)
    item_ids = data['click_article_id'].unique()

    train_set = []
    test_set = []
    for reviewerID, hist in tqdm(data.groupby('user_id'), disable=not logger.isEnabledFor(logging.DEBUG)):
        pos_list = hist['click_article_id'].tolist()
        
        if negsample > 0:
            candidate_set = list(set(item_ids) - set(pos_list))   # 用户没看过的文章里面选择负样本
            neg_list = np.random.choice(candidate_set,size=len(pos_list)*negsample,replace=True)  # 对于每个正样本，选择n个负样本
            
        # 长度只有一个的时候，需要把这条数据也放到训练集中，不然的话最终学到的embedding就会有缺失
        if len(pos_list) == 1:
            train_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
            test_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
            
        # 滑窗构造正负样本
        for i in range(1, len(pos_list)):
            hist = pos_list[:i]
            
            if i != len(pos_list) - 1:
                train_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))  # 正样本 [user_id, his_item, pos_item, label, len(his_item)]
                for negi in range(negsample):
                    train_set.append((reviewerID, hist[::-1], neg_list[i*negsample+negi], 0,len(hist[::-1]))) # 负样本 [user_id, his_item, neg_item, label, len(his_item)]
            else:
                # 将最长的那一个序列长度作为测试数据
                test_set.append((reviewerID, hist[::-1], pos_list[i],1,len(hist[::-1])))
                
    random.shuffle(train_set)
    random.shuffle(test_set)
    
    return train_set, test_set

# 将输入的数据进行padding，使得序列特征的长度都一致
def gen_model_input(train_set,user_profile,seq_max_len):

    train_uid = np.array([line[0] for line in train_set])
    train_seq = [line[1] for line in train_set]
    train_iid = np.array([line[2] for line in train_set])
    train_label = np.array([line[3] for line in train_set])
    train_hist_len = np.array([line[4] for line in train_set])

    train_seq_pad = pad_sequences(train_seq, maxlen=seq_max_len, padding='post', truncating='post', value=0)
    train_model_input = {"user_id": train_uid, "click_article_id": train_iid, "hist_article_id": train_seq_pad,
                         "hist_len": train_hist_len}

    return train_model_input, train_label


### itemCF recall

上面已经通过协同过滤，Embedding检索的方式得到了文章的相似度矩阵，下面使用协同过滤的思想，给用户召回与其历史文章相似的文章。
这里在召回的时候，也是用了关联规则的方式：

1. 考虑相似文章与历史点击文章顺序的权重(细节看代码)
2. 考虑文章创建时间的权重，也就是考虑相似文章与历史点击文章创建时间差的权重
3. 考虑文章内容相似度权重(使用Embedding计算相似文章相似度，但是这里需要注意，在Embedding的时候并没有计算所有商品两两之间的相似度，所以相似的文章与历史点击文章不存在相似度，需要做特殊处理)

In [ ]:
# 基于商品的召回i2i
def item_based_recommend(user_id, user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim):
    """
        基于文章协同过滤的召回
        :param user_id: 用户id
        :param user_item_time_dict: 字典, 根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
        :param i2i_sim: 字典，文章相似性矩阵
        :param sim_item_topk: 整数， 选择与当前文章最相似的前k篇文章
        :param recall_item_num: 整数， 最后的召回文章数量
        :param item_topk_click: 列表，点击次数最多的文章列表，用户召回补全
        :param emb_i2i_sim: 字典基于内容embedding算的文章相似矩阵
        
        return: 召回的文章列表 {item1:score1, item2: score2...}
        
    """
    # 获取用户历史交互的文章
    user_hist_items = user_item_time_dict[user_id]
    
    item_rank = {}
    for loc, (i, click_time) in enumerate(user_hist_items):
        if i not in i2i_sim:
            continue
        for j, wij in sorted(i2i_sim[i].items(), key=lambda x: x[1], reverse=True)[:sim_item_topk]:
            if j in user_hist_items:
                continue
            
            # 文章创建时间差权重
            created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
            # 相似文章和历史点击文章序列中历史文章所在的位置权重
            loc_weight = (0.9 ** (len(user_hist_items) - loc))
            
            content_weight = 1.0
            if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                content_weight += emb_i2i_sim[i][j]
            if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                content_weight += emb_i2i_sim[j][i]
                
            item_rank.setdefault(j, 0)
            item_rank[j] += created_time_weight * loc_weight * content_weight * wij
    
    # 不足10个，用热门商品补全
    if len(item_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in item_rank.items(): # 填充的item应该不在原来的列表中
                continue
            item_rank[item] = - i - 100 # 随便给个负数就行
            if len(item_rank) == recall_item_num:
                break
    
    item_rank = sorted(item_rank.items(), key=lambda x: x[1], reverse=True)[:recall_item_num]
        
    return item_rank


#### ItemCF sim 召回

### ⚠️ 数据泄漏防护（召回阶段）

| 模式 | 用户历史点击来源 | 目标用户 | 说明 |
|------|-----------------|---------|------|
| 离线模式 | `click_val` (验证用户历史) | 验证用户 | ⚠️ 不能用 `val_ans` |
| 在线模式 | `click_trn` (全量) | 测试集用户 | - |

In [ ]:
# ============================================================================
# ItemCF 召回
# ============================================================================
# ⚠️ 离线模式：
#    - 相似度矩阵：来自 click_hist_all（全用户历史）
#    - 用户历史：来自 click_hist_all（全用户历史）
#    - 评估：用 click_last_all
# ⚠️ 在线模式：
#    - 使用全量数据训练相似度矩阵
#    - 为测试用户生成召回
# ============================================================================

print("=" * 60)
print("📌 ItemCF 召回")

if MODE == "offline":
    print("🔬 [离线模式] 为全用户 leave-last-out 生成召回...")
    user_item_time_dict = get_user_item_time(click_hist_all)
    target_users = click_hist_all['user_id'].unique()
else:
    print("🚀 [在线模式] 为测试集用户生成召回...")
    user_item_time_dict = get_user_item_time(click_trn)
    target_users = click_tst['user_id'].unique()

# 加载相似度矩阵
i2i_sim = pickle.load(open(save_path / 'itemcf_i2i_sim_all.pkl', 'rb'))
emb_i2i_sim = pickle.load(open(save_path / 'emb_i2i_sim_all.pkl', 'rb'))

sim_item_topk = 20
recall_item_num = 150
item_topk_click = get_item_topk_click(click_trn, k=50)

user_recall_items_dict = defaultdict(dict)
for user in tqdm(target_users, desc="ItemCF召回", disable=not logger.isEnabledFor(logging.DEBUG)):
    if user in user_item_time_dict:
        user_recall_items_dict[user] = item_based_recommend(
            user, user_item_time_dict, i2i_sim, sim_item_topk,
            recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim,
        )

user_multi_recall_dict['itemcf_sim_itemcf_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['itemcf_sim_itemcf_recall'], open(save_path / 'itemcf_recall_dict_all.pkl', 'wb'))

if MODE == "offline":
    print("\n📊 ItemCF 召回效果评估:")
    metrics_recall(user_multi_recall_dict['itemcf_sim_itemcf_recall'], trn_last_click_df, topk=150)
    print_recall_details(user_multi_recall_dict['itemcf_sim_itemcf_recall'], trn_last_click_df, item_info_df)

print("=" * 60)


#### embedding sim 召回

In [ ]:
# ============================================================================
# Embedding 相似度召回
# ============================================================================
# ⚠️ 同样遵循 All 协议：离线用全用户历史，在线用测试用户历史作为 query
# ============================================================================

print("=" * 60)
print("📌 Embedding 相似度召回")

if MODE == "offline":
    print("🔬 [离线模式] 为全用户 leave-last-out 生成召回...")
    user_item_time_dict = get_user_item_time(click_hist_all)
    target_users = click_hist_all['user_id'].unique()
else:
    print("🚀 [在线模式] 为测试集用户生成召回...")
    user_item_time_dict = get_user_item_time(click_trn)
    target_users = click_tst['user_id'].unique()

i2i_sim = pickle.load(open(save_path / 'emb_i2i_sim_all.pkl','rb'))

sim_item_topk = 20
recall_item_num = 150
item_topk_click = get_item_topk_click(click_trn, k=50)

user_recall_items_dict = defaultdict(dict)
for user in tqdm(target_users, desc="Embedding召回", disable=not logger.isEnabledFor(logging.DEBUG)):
    if user in user_item_time_dict:
        user_recall_items_dict[user] = item_based_recommend(
            user, user_item_time_dict, i2i_sim, sim_item_topk,
            recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim,
        )

user_multi_recall_dict['embedding_sim_item_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['embedding_sim_item_recall'], open(save_path / 'embedding_sim_item_recall_all.pkl', 'wb'))

if MODE == "offline":
    print("\n📊 Embedding 召回效果评估:")
    metrics_recall(user_multi_recall_dict['embedding_sim_item_recall'], trn_last_click_df, topk=150)
    print_recall_details(user_multi_recall_dict['embedding_sim_item_recall'], trn_last_click_df, item_info_df)

print("=" * 60)


### userCF召回

基于用户协同过滤，核心思想是给用户推荐与其相似的用户历史点击文章，因为这里涉及到了相似用户的历史文章，这里仍然可以加上一些关联规则来给用户可能点击的文章进行加权，这里使用的关联规则主要是考虑相似用户的历史点击文章与被推荐用户历史点击商品的关系权重，而这里的关系就可以直接借鉴基于物品的协同过滤相似的做法，只不过这里是对被推荐物品关系的一个累加的过程，下面是使用的一些关系权重，及相关的代码：

1. 计算被推荐用户历史点击文章与相似用户历史点击文章的相似度，文章创建时间差，相对位置的总和，作为各自的权重

In [ ]:
# 基于用户的召回 u2u2i
def user_based_recommend(user_id, user_item_time_dict, u2u_sim, sim_user_topk, recall_item_num, 
                         item_topk_click, item_created_time_dict, emb_i2i_sim):
    """
        基于文章协同过滤的召回
        :param user_id: 用户id
        :param user_item_time_dict: 字典, 根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
        :param u2u_sim: 字典，文章相似性矩阵
        :param sim_user_topk: 整数， 选择与当前用户最相似的前k个用户
        :param recall_item_num: 整数， 最后的召回文章数量
        :param item_topk_click: 列表，点击次数最多的文章列表，用户召回补全
        :param item_created_time_dict: 文章创建时间列表
        :param emb_i2i_sim: 字典基于内容embedding算的文章相似矩阵
        
        return: 召回的文章列表 {item1:score1, item2: score2...}
    """
    # 历史交互
    user_item_time_list = user_item_time_dict[user_id]    # {item1: time1, item2: time2...}
    user_hist_items = set([i for i, t in user_item_time_list])   # 存在一个用户与某篇文章的多次交互， 这里得去重
    
    items_rank = {}
    for sim_u, wuv in sorted(u2u_sim[user_id].items(), key=lambda x: x[1], reverse=True)[:sim_user_topk]:
        for i, click_time in user_item_time_dict[sim_u]:
            if i in user_hist_items:
                continue
            items_rank.setdefault(i, 0)
            
            loc_weight = 1.0
            content_weight = 1.0
            created_time_weight = 1.0
            
            # 当前文章与该用户看的历史文章进行一个权重交互
            for loc, (j, click_time) in enumerate(user_item_time_list):
                # 点击时的相对位置权重
                loc_weight += 0.9 ** (len(user_item_time_list) - loc)
                # 内容相似性权重
                if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                    content_weight += emb_i2i_sim[i][j]
                if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                    content_weight += emb_i2i_sim[j][i]
                
                # 创建时间差权重
                created_time_weight += np.exp(0.8 * np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                
            items_rank[i] += loc_weight * content_weight * created_time_weight * wuv
        
    # 热度补全
    if len(items_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in items_rank.items(): # 填充的item应该不在原来的列表中
                continue
            items_rank[item] = - i - 100 # 随便给个复数就行
            if len(items_rank) == recall_item_num:
                break
        
    items_rank = sorted(items_rank.items(), key=lambda x: x[1], reverse=True)[:recall_item_num]    
    
    return items_rank


#### userCF sim召回

#### user embedding sim召回

虽然没有直接跑usercf的计算用户之间的相似度，为了验证上述基于用户的协同过滤的代码，下面使用了YoutubeDNN过程中产生的user embedding来进行向量检索每个user最相似的topk个user，在使用这里得到的u2u的相似性矩阵，使用usercf进行召回，具体代码如下

通过YoutubeDNN得到的user_embedding

## 冷启动问题

**冷启动问题可以分成三类：文章冷启动，用户冷启动，系统冷启动。**

- 文章冷启动：对于一个平台系统新加入的文章，该文章没有任何的交互记录，如何推荐给用户的问题。(对于我们场景可以认为是，日志数据中没有出现过的文章都可以认为是冷启动的文章)
- 用户冷启动：对于一个平台系统新来的用户，该用户还没有文章的交互信息，如何给该用户进行推荐。(对于我们场景就是，测试集中的用户是否在测试集对应的log数据中出现过，如果没有出现过，那么可以认为该用户是冷启动用户。但是有时候并没有这么严格，我们也可以自己设定某些指标来判别哪些用户是冷启动用户，比如通过使用时长，点击率，留存率等等)
- 系统冷启动：就是对于一个平台刚上线，还没有任何的相关历史数据，此时就是系统冷启动，其实也就是前面两种的一个综合。

**当前场景下冷启动问题的分析：**

对当前的数据进行分析会发现，日志中所有出现过的点击文章只有3w多个，而整个文章库中却有30多万，那么测试集中的用户最后一次点击是否会点击没有出现在日志中的文章呢？如果存在这种情况，说明用户点击的文章之前没有任何的交互信息，这也就是我们所说的文章冷启动。通过数据分析还可以发现，测试集用户只有一次点击的数据占得比例还不少，其实仅仅通过用户的一次点击就给用户推荐文章使用模型的方式也是比较难的，这里其实也可以考虑用户冷启动的问题，但是这里只给出物品冷启动的一些解决方案及代码，关于用户冷启动的话提一些可行性的做法。

1. 文章冷启动(没有冷启动的探索问题)    
   其实我们这里不是为了做文章的冷启动而做冷启动，而是猜测用户可能会点击一些没有在log数据中出现的文章，我们要做的就是如何从将近27万的文章中选择一些文章作为用户冷启动的文章，这里其实也可以看成是一种召回策略，我们这里就采用简单的比较好理解的基于规则的召回策略来获取用户可能点击的未出现在log数据中的文章。
   现在的问题变成了：如何给每个用户考虑从27万个商品中获取一小部分商品？随机选一些可能是一种方案。下面给出一些参考的方案。
   1. 首先基于Embedding召回一部分与用户历史相似的文章
   2. 从基于Embedding召回的文章中通过一些规则过滤掉一些文章，使得留下的文章用户更可能点击。我们这里的规则，可以是，留下那些与用户历史点击文章主题相同的文章，或者字数相差不大的文章。并且留下的文章尽量是与测试集用户最后一次点击时间更接近的文章，或者是当天的文章也行。
2. 用户冷启动    
   这里对测试集中的用户点击数据进行分析会发现，测试集中有百分之20的用户只有一次点击，那么这些点击特别少的用户的召回是不是可以单独做一些策略上的补充呢？或者是在排序后直接基于规则加上一些文章呢？这些都可以去尝试，这里没有提供具体的做法。
   

**注意：**   

这里看似和基于embedding计算的item之间相似度然后做itemcf是一致的，但是现在我们的目的不一样，我们这里的目的是找到相似的向量，并且还没有出现在log日志中的商品，再加上一些其他的冷启动的策略，这里需要找回的数量会偏多一点，不然被筛选完之后可能都没有文章了

In [ ]:
# 先进行 embedding-sim 召回，这里不做评估，只作为冷启动候选池
if MODE == "offline":
    trn_hist_click_df = click_hist_all
else:
    trn_hist_click_df = click_tst

user_recall_items_dict = defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)
i2i_sim = pickle.load(open(save_path / 'emb_i2i_sim_all.pkl','rb'))

sim_item_topk = 150
recall_item_num = 150  # 稍微召回多一点文章，便于后续的规则筛选

item_topk_click = get_item_topk_click(click_trn, k=50)
for user in tqdm(trn_hist_click_df['user_id'].unique(), disable=not logger.isEnabledFor(logging.DEBUG)):
    user_recall_items_dict[user] = item_based_recommend(
        user, user_item_time_dict, i2i_sim, sim_item_topk,
        recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim,
    )
pickle.dump(user_recall_items_dict, open(save_path / 'cold_start_items_raw_dict_all.pkl', 'wb'))


In [ ]:
# 基于规则进行文章过滤
# 保留文章主题与用户历史浏览主题相似的文章
# 保留文章字数与用户历史浏览文章字数相差不大的文章
# 保留最后一次点击当天的文章
# 按照相似度返回最终的结果

def get_click_article_ids_set(all_click_df):
    return set(click_trn.click_article_id.values)

def cold_start_items(user_recall_items_dict, user_hist_item_typs_dict, user_hist_item_words_dict, \
                     user_last_item_created_time_dict, item_type_dict, item_words_dict, 
                     item_created_time_dict, click_article_ids_set, recall_item_num):
    """
        冷启动的情况下召回一些文章
        :param user_recall_items_dict: 基于内容embedding相似性召回来的很多文章， 字典， {user1: [item1, item2, ..], }
        :param user_hist_item_typs_dict: 字典， 用户点击的文章的主题映射
        :param user_hist_item_words_dict: 字典， 用户点击的历史文章的字数映射
        :param user_last_item_created_time_idct: 字典，用户点击的历史文章创建时间映射
        :param item_tpye_idct: 字典，文章主题映射
        :param item_words_dict: 字典，文章字数映射
        :param item_created_time_dict: 字典， 文章创建时间映射
        :param click_article_ids_set: 集合，用户点击过得文章, 也就是日志里面出现过的文章
        :param recall_item_num: 召回文章的数量， 这个指的是没有出现在日志里面的文章数量
    """

    cold_start_user_items_dict = {}
    for user, item_list in tqdm(user_recall_items_dict.items(), disable=not logger.isEnabledFor(logging.DEBUG)):
        cold_start_user_items_dict.setdefault(user, [])
        for item, score in item_list:
            # 获取历史文章信息
            hist_item_type_set = user_hist_item_typs_dict[user]
            hist_mean_words = user_hist_item_words_dict[user]
            hist_last_item_created_time = user_last_item_created_time_dict[user]
            hist_last_item_created_time = datetime.fromtimestamp(hist_last_item_created_time)

            # 获取当前召回文章的信息
            curr_item_type = item_type_dict[item]
            curr_item_words = item_words_dict[item]
            curr_item_created_time = item_created_time_dict[item]
            curr_item_created_time = datetime.fromtimestamp(curr_item_created_time)

            # 首先，文章不能出现在用户的历史点击中， 然后根据文章主题，文章单词数，文章创建时间进行筛选
            if curr_item_type not in hist_item_type_set or \
                item in click_article_ids_set or \
                abs(curr_item_words - hist_mean_words) > 200 or \
                abs((curr_item_created_time - hist_last_item_created_time).days) > 90: 
                continue

            cold_start_user_items_dict[user].append((item, score))      # {user1: [(item1, score1), (item2, score2)..]...}

    # 需要控制一下冷启动召回的数量
    cold_start_user_items_dict = {k: sorted(v, key=lambda x:x[1], reverse=True)[:recall_item_num] \
                                  for k, v in cold_start_user_items_dict.items()}

    pickle.dump(cold_start_user_items_dict, open(save_path / 'cold_start_user_items_dict_all.pkl', 'wb'))

    return cold_start_user_items_dict


In [ ]:
all_click_df_ = all_click_df.copy()
all_click_df_ = all_click_df_.merge(item_info_df, how='left', on='click_article_id')
user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict = get_user_hist_item_info_dict(all_click_df_)
click_article_ids_set = get_click_article_ids_set(all_click_df)
# 需要注意的是
# 这里使用了很多规则来筛选冷启动的文章，所以前面再召回的阶段就应该尽可能的多召回一些文章，否则很容易被删掉
cold_start_user_items_dict = cold_start_items(
    user_recall_items_dict,
    user_hist_item_typs_dict,
    user_hist_item_words_dict,
    user_last_item_created_time_dict,
    item_type_dict,
    item_words_dict,
    item_created_time_dict,
    click_article_ids_set,
    recall_item_num,
)

user_multi_recall_dict['cold_start_recall'] = cold_start_user_items_dict
pickle.dump(user_multi_recall_dict['cold_start_recall'], open(save_path / 'cold_start_recall_all.pkl', 'wb'))

if MODE == "offline":
    print_recall_details(user_multi_recall_dict['cold_start_recall'], trn_last_click_df, item_info_df)
else:
    print(f"✅ 冷启动召回结果已保存: {save_path / 'cold_start_recall_all.pkl'}")


## 多路召回合并

多路召回合并就是将前面所有的召回策略得到的用户文章列表合并起来，下面是对前面所有召回结果的汇总
1. 基于itemcf计算的item之间的相似度sim进行的召回 
2. 基于embedding搜索得到的item之间的相似度进行的召回
3. YoutubeDNN召回
4. YoutubeDNN得到的user之间的相似度进行的召回
5. 基于冷启动策略的召回

**注意：**  
在做召回评估的时候就会发现有些召回的效果不错有些召回的效果很差，所以对每一路召回的结果，我们可以认为的定义一些权重，来做最终的相似度融合

In [ ]:
def combine_recall_results(user_multi_recall_dict, weight_dict=None, topk=25):
    final_recall_items_dict = {}

    def norm_user_recall_items_sim(sorted_item_list):
        if len(sorted_item_list) < 2:
            return sorted_item_list

        min_sim = sorted_item_list[-1][1]
        max_sim = sorted_item_list[0][1]

        norm_sorted_item_list = []
        for item, score in sorted_item_list:
            if max_sim > 0:
                norm_score = 1.0 * (score - min_sim) / (max_sim - min_sim) if max_sim > min_sim else 1.0
            else:
                norm_score = 0.0
            norm_sorted_item_list.append((item, norm_score))

        return norm_sorted_item_list

    print('多路召回合并...')
    for method, user_recall_items in tqdm(user_multi_recall_dict.items(), disable=not logger.isEnabledFor(logging.DEBUG)):
        print(method + '...')
        recall_method_weight = 1 if weight_dict is None else weight_dict[method]

        for user_id, sorted_item_list in user_recall_items.items():
            user_recall_items[user_id] = norm_user_recall_items_sim(sorted_item_list)

        for user_id, sorted_item_list in user_recall_items.items():
            final_recall_items_dict.setdefault(user_id, {})
            for item, score in sorted_item_list:
                final_recall_items_dict[user_id].setdefault(item, 0)
                final_recall_items_dict[user_id][item] += recall_method_weight * score

    final_recall_items_dict_rank = {}
    for user, recall_item_dict in final_recall_items_dict.items():
        final_recall_items_dict_rank[user] = sorted(
            recall_item_dict.items(), key=lambda x: x[1], reverse=True
        )[:topk]

    pickle.dump(final_recall_items_dict_rank, open(save_path / FINAL_RECALL_DICT_NAME, 'wb'))
    return final_recall_items_dict_rank


In [ ]:
# ============================================================================
# 多路召回合并
# ============================================================================

print("=" * 60)
print("📌 多路召回合并")

# 权重配置
weight_dict = {
    'itemcf_sim_itemcf_recall': 1.0,
    'embedding_sim_item_recall': 1.0,
    'cold_start_recall': 1.0,
}

# 合并召回结果
final_recall_items_dict_rank = combine_recall_results(user_multi_recall_dict, weight_dict, topk=150)

if MODE == "offline":
    print("\n📊 ========== 最终合并召回效果评估 ==========")
    metrics_recall(final_recall_items_dict_rank, trn_last_click_df, topk=150)
    print(f"✅ 离线评估完成，最终召回已保存: {save_path / FINAL_RECALL_DICT_NAME}")
else:
    print(f"✅ 在线模式：召回结果已生成，已保存: {save_path / FINAL_RECALL_DICT_NAME}")

print("=" * 60)


## 📋 使用说明总结

### 工作流程

1. **开发调参阶段** (离线模式)
   ```python
   MODE = "offline"
   DEBUG_MODE = True  # 可选，加速调试
   ```
   - 运行全部 Cell，查看各路召回的 Hit Rate
   - 调整参数（如 `sim_item_topk`、`recall_item_num`、`weight_dict`）
   - 重复运行直到效果满意

2. **全量验证阶段** (离线模式 + 全量数据)
   ```python
   MODE = "offline"
   DEBUG_MODE = False
   ```
   - 确认全量数据上的召回效果

3. **生成最终召回结果** (在线模式)
   ```python
   MODE = "online"
   DEBUG_MODE = False
   ```
   - 运行全部 Cell，生成 `final_recall_for_rank.csv`

---

### ⚠️ 数据泄漏防护检查清单

| 检查项 | 离线模式要求 | ✓ |
|--------|-------------|---|
| 相似度计算 | 只用 `click_trn`（训练用户） | ☑️ |
| 用户历史点击 | 只用 `click_val`（不含最后一次） | ☑️ |
| YoutubeDNN 训练 | 只用 `click_trn` | ☑️ |
| 评估 ground truth | 用 `val_ans`（最后一次点击） | ☑️ |

---

### 关键参数说明

| 参数 | 说明 | 建议值 |
|------|------|--------|
| `sim_item_topk` | ItemCF 相似物品数量 | 10-30 |
| `recall_item_num` | 每路召回文章数量 | 10-50 |
| `weight_dict` | 各路召回权重 | 根据 Hit Rate 调整 |
| `topk` (合并) | 最终召回数量 | 100-200 |

---

## 总结

上述实现了如下召回策略：

1. 基于关联规则的itemcf
2. 基于关联规则的usercf
3. youtubednn召回
4. 冷启动召回

对于上述实现的召回策略其实都不是最优的结果，我们只是做了个简单的尝试，其中还有很多地方可以优化，包括已经实现的这些召回策略的参数或者新加一些，修改一些关联规则都可以。当然还可以尝试更多的召回策略，比如对新闻进行热度召回等等。

In [ ]:
# ============================================================================
# 保存最终召回结果
# ============================================================================

print("=" * 60)
print("📌 保存召回结果")

rows = [(u, iid, s) for u, items in final_recall_items_dict_rank.items() for iid, s in items]
pred_data = pd.DataFrame(rows, columns=['user_id', 'click_article_id', 'score'])
pred_data.to_csv(save_path / FINAL_RECALL_CSV_NAME, index=False)

print(f"✅ 召回结果已保存: {save_path / FINAL_RECALL_CSV_NAME}")
print(f"✅ 召回字典已保存: {save_path / FINAL_RECALL_DICT_NAME}")
print(f"   用户数: {pred_data['user_id'].nunique()}")
print(f"   召回总数: {len(pred_data)}")

if MODE == "offline":
    print("\n📋 下一步：")
    print("   1. 运行 5.feature_engineering_all.ipynb 生成 _all 排序特征")
    print("   2. 运行 jrc-ranking_all.ipynb 验证 JRC 新链路")
else:
    print("\n📋 下一步：")
    print("   1. 运行 5.feature_engineering_all.ipynb 进行特征工程")
    print("   2. 运行 jrc-ranking_all.ipynb 进行排序与提交")

print("=" * 60)
